# Week1_ex1 - Torque on a Rotated Permanent Magnet

This exercise looks at the torque acting on an NdFe35 bar magnet placed next to a current-carrying ring coil, rotated 45° so there's an actual angle for the torque to act on, using the Magnetostatic solver in Maxwell 3D.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [ ]:
## Initialize AEDT Desktop session and create Maxwell 3D project/design ##

# Start AEDT session (specify version and student license flag)
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# Disable autosave to prevent popup interruptions during scripted runs
DT.disable_autosave()

# Solution type: Magnetostatic
sol_type = "Magnetostatic"

# Create Maxwell 3D design object
# On first creation, a new project and design are generated with default names
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)
# Reference to odesign, used for AEDT's built-in recording-style API calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

# Project name (change freely as needed)
proj_name = "Week1_ex1"

# Save directory driven by the ANSYS_PROJECT_DIR environment variable (portable, no hardcoding)
dir = os.path.join(os.environ["ANSYS_PROJECT_DIR"], proj_name)
print(dir)

# Create directory if it doesn't exist yet (safe to re-run; won't error if already there)
os.makedirs(dir, exist_ok=True)

# Design name
desi_name = "Week1_ex1"

In [ ]:
## Save project and apply design name ##

# Reference to the project object containing this design
proj = M3D.oproject

# Save project with the target file name (directory created above)
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

# Rename design to match the target design name
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
## Create geometry ##

# Sketch a circle and sweep it around an axis to form a ring-shaped coil

origin = [0, 5, 0]
coil = M3D.modeler.create_circle(orientation="XY", origin=origin, radius=0.5, num_sides=12, is_covered=True, name="Coil", material=None, non_model=False)

M3D.modeler.sweep_around_axis(assignment=coil, axis="X", sweep_angle=360, draft_angle=0, number_of_segments=30)

M3D.assign_material(assignment=coil, material="copper")

# Create the bar-shaped permanent magnet
origin = [-3, -0.5, -0.5]
sizes = [6, 1, 1]
magnet = M3D.modeler.create_box(origin, sizes, name="Magnet", material="NdFe35")


In [ ]:
## Check magnet material (NdFe35) magnetization direction / coercivity ##

NdFe35 = M3D.materials.exists_material(material="NdFe35")

display(NdFe35.get_magnetic_coercivity())

# # Use this if the magnetization direction differs from expected (uncomment if needed)
# NdFe35.set_magnetic_coercivity(value='-890000A_per_meter', x="1", y="0", z="0")

In [ ]:
## Create coil terminal cross-sections for current excitation ##

coil_section = []
coil = [coil]   # Wrap in a list to allow loop processing
for c in coil : 
    M3D.modeler.section(assignment=c, plane="XY", create_new=True, section_cross_object=False)
    coil_section.append( M3D.modeler.sheet_objects[-1] )

M3D.modeler.split(assignment=coil_section, plane="ZX", sides="NegativeOnly", tool=None, split_crossing_objs=False, delete_invalid_objs=True)

# Assign current excitation to each coil cross-section

coil_terminal = []
for s in coil_section :
    coil_terminal.append( M3D.assign_current(assignment=s, amplitude="100A", phase='0deg', solid=False, swap_direction=False, name=None) )


In [ ]:
## Assign virtual torque boundary condition to the magnet ##

M3D.assign_torque(assignment=magnet, coordinate_system='Global', is_positive=True, is_virtual=True, axis='Z', torque_name="Torque1")


In [ ]:
## Rotate coil to set relative angle with the magnet ##

 
M3D.modeler.rotate(assignment=coil, axis="Z", angle=45.0, units='deg')
M3D.modeler.rotate(assignment=coil_section, axis="Z", angle=45.0, units='deg')

In [ ]:
## Create simulation region (surrounding air/vacuum domain) ##

region = M3D.modeler.create_region(pad_value=100, pad_type='Percentage Offset', name='Region')

In [ ]:
## Create and inspect analysis setup ##

# Create analysis setup object
my_setup = M3D.create_setup(name="Setup1")

# Check available setup properties for the current solution type (stored as a dict)
display(my_setup.props)

In [ ]:
# Modify desired properties from the dict above (maximum number of adaptive passes)

my_setup.props['MaximumPasses'] = 10


In [ ]:
# Run the analysis
my_setup.analyze()

In [ ]:
# Save final results
M3D.save_project()